# Rule-Consistency Auditor
- Francesco Buda, francesco.buda3@studio.unibo.it
- Emanuele Sanchi, emanuele.sanchi@studio.unibo.it
- Tommaso Severi, tommaso.severi2@studio.unibo.it

## Import libraries

In [1]:
import time
import carla_utils
import data_list_bind
import carla
import pygame
import datetime
import json
import threading

pygame 2.6.1 (SDL 2.28.4, Python 3.7.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Setup log file and CARLA world

In [2]:
log_filename = f"logs/rca_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
world, spectator, client = carla_utils.world_connect()

RuntimeError: time-out of 10000ms while waiting for the simulator, make sure the simulator is ready and connected to localhost:2000

## Spawn vehicles

In [3]:
vehicle_autopilot = carla_utils.spawn_random_vehicle_no_bike(world, spawn_index=0, autopilot=True)
ego_vehicle = carla_utils.spawn_random_vehicle_no_bike(world, spawn_index=1, autopilot=False)

## Setup binder thread

In [4]:
binder = data_list_bind.VariableBinder(world, ego_vehicle)
scene_data_shared = {}
scene_data_lock = threading.Lock()
binder_stop_event = threading.Event()

def binder_thread_fn():
    while not binder_stop_event.is_set():
        data = binder.compute_scene_data()
        with scene_data_lock:
            scene_data_shared.update(data)
        time.sleep(0.05)

binder_thread = threading.Thread(target=binder_thread_fn, daemon=True)
binder_thread.start()

## Main control loop

In [ ]:
pygame.init()
screen = pygame.display.set_mode((300, 80))
pygame.display.set_caption("RCA Control  |  WASD · R=reverse · Q=quit")

control        = carla.VehicleControl()
throttle_step  = 0.03
steer_step     = 0.04
brake_step     = 0.1
running        = True
frame_count    = 0
reverse        = False
log_buffer     = []
FLUSH_EVERY    = 20

tick_event = threading.Event()

def on_server_tick(snapshot):
    tick_event.set()

world.on_tick(on_server_tick)

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

try:
    while running:
        tick_event.wait(timeout=0.1)
        tick_event.clear()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                elif event.key == pygame.K_r:
                    reverse = not reverse

        keys = pygame.key.get_pressed()

        if keys[pygame.K_w]:
            control.throttle = clamp(control.throttle + throttle_step, 0.0, 1.0)
            control.brake = 0.0
        elif keys[pygame.K_s]:
            control.brake = clamp(control.brake + brake_step, 0.0, 1.0)
            control.throttle = 0.0
        else:
            control.throttle = 0.0
            control.brake = 0.0

        if keys[pygame.K_a]:
            control.steer = clamp(control.steer - steer_step, -1.0, 1.0)
        elif keys[pygame.K_d]:
            control.steer = clamp(control.steer + steer_step, -1.0, 1.0)
        else:
            control.steer -= control.steer * 0.1

        control.reverse = reverse
        ego_vehicle.apply_control(control)

        carla_utils.move_spectator_to(
            ego_vehicle.get_transform(), spectator,
            distance=4.0, z=2.0, pitch=-10
        )

        with scene_data_lock:
            scene_data = dict(scene_data_shared)
        timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]

        log_entry = {
            'frame':     frame_count,
            'timestamp': timestamp,
            'reverse':   reverse,
            'control': {
                'throttle': round(float(control.throttle), 3),
                'brake':    round(float(control.brake),    3),
                'steer':    round(float(control.steer),    3),
            },
            'scene_data': {
                k: str(v) if hasattr(v, '__dict__') else v
                for k, v in scene_data.items()
            }
        }
        log_buffer.append(log_entry)

        if len(log_buffer) >= FLUSH_EVERY:
            with open(log_filename, 'a') as f:
                for entry in log_buffer:
                    f.write(json.dumps(entry) + '\n')
            log_buffer.clear()

        frame_count += 1

finally:
    world.on_tick(None)
    binder_stop_event.set()
    binder_thread.join(timeout=2.0)
    
    if log_buffer:
        with open(log_filename, 'a') as f:
            for entry in log_buffer:
                f.write(json.dumps(entry) + '\n')

    try:
        ego_vehicle.destroy()
    except Exception:
        pass
    try:
        vehicle_autopilot.destroy()
    except Exception:
        pass

    pygame.quit()
    print(f"Simulazione terminata dopo {frame_count} frame. Log: {log_filename}")

TypeError: callback argument must be callable!

: 

## Cleanup

In [ ]:
print("Destroying all vehicles in the simulator...")
for actor in world.get_actors().filter('vehicle.*'):
    try:
        actor.destroy()
        print(f"  Destroyed: {actor.id}")
    except Exception as e:
        print(f"  Skip {actor.id}: {e}")
print("All vehicles cleaned up. Ready for next run!")

Destroying all vehicles in the simulator...
All vehicles cleaned up. Ready for next run!


: 